# AutoVoice: Exploratory Data Analysis (NHTSA Complaints)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/devdasrahul/autovoice/blob/main/notebooks/01_eda.ipynb)

This notebook inspects public NHTSA safety defect complaint records, calculates length distributions, evaluates component/make frequencies, and analyzes physical hazard indicators.

**Google Colab Ready:**
- If running in Colab, it will automatically prompt you to upload `complaints.parquet` or `nhtsa_complaints_raw.csv`.
- When execution completes, all generated charts and `eda_summary.json` are packaged and **automatically downloaded** to your machine.


In [ ]:
# 1. Environment Setup
import os, sys, json, shutil
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    from google.colab import files
    IN_COLAB = True
    print('🚀 Running inside Google Colab environment.')
except ImportError:
    IN_COLAB = False
    print('💻 Running in standard local Python environment.')


In [ ]:
# 2. Interactive Data Upload / Loading
data_file = None
candidate_paths = [
    Path('complaints.parquet'),
    Path('data/processed/complaints.parquet'),
    Path('../data/processed/complaints.parquet'),
    Path('nhtsa_complaints_raw.csv'),
    Path('data/raw/nhtsa_complaints_raw.csv'),
]

for cp in candidate_paths:
    if cp.exists():
        data_file = cp
        break

if data_file is None and IN_COLAB:
    print('📁 Data file not found locally. Please upload complaints.parquet or nhtsa_complaints_raw.csv:')
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(('.parquet', '.csv')):
            data_file = Path(fname)
            break

if data_file is None:
    raise FileNotFoundError('No complaint data file found. Please upload complaints.parquet or run make data.')

print(f'✅ Loading data from: {data_file}')
if str(data_file).endswith('.parquet'):
    df = pd.read_parquet(data_file)
else:
    df = pd.read_csv(data_file)
    # normalize column names
    col_map = {'CDESCR': 'narrative', 'MAKE': 'make', 'MODEL': 'model', 'YEAR': 'model_year', 'COMPNAME': 'component'}
    df = df.rename(columns={k: v for k, v in col_map.items() if k in df.columns})

text_col = 'narrative' if 'narrative' in df.columns else ('cdescr' if 'cdescr' in df.columns else df.columns[0])
print(f'Corpus size: {len(df)} rows. Target text column: {text_col}')
df.head(2)


In [ ]:
# 3. Narrative Length Distribution Analysis
output_dir = Path('eda_artifacts')
output_dir.mkdir(parents=True, exist_ok=True)

df['char_length'] = df[text_col].fillna('').astype(str).str.len()
df['word_count'] = df[text_col].fillna('').astype(str).str.split().str.len()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
df['char_length'].hist(bins=35, ax=ax1, color='#0284c7', edgecolor='black', alpha=0.8)
ax1.set_title('Narrative Character Length Distribution')
ax1.set_xlabel('Character Count')
ax1.set_ylabel('Number of Complaints')

df['word_count'].hist(bins=35, ax=ax2, color='#ea580c', edgecolor='black', alpha=0.8)
ax2.set_title('Narrative Word Count Distribution')
ax2.set_xlabel('Word Count')

plt.tight_layout()
fig.savefig(output_dir / 'length_distribution.png', dpi=150)
plt.show()

print(df[['char_length', 'word_count']].describe())


In [ ]:
# 4. Top Vehicle Makes & Defective Components
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

if 'make' in df.columns:
    df['make'].value_counts().head(10).plot(kind='barh', ax=ax1, color='#3b82f6')
    ax1.set_title('Top 10 Reported Vehicle Makes')
    ax1.invert_yaxis()

if 'component' in df.columns:
    df['component'].value_counts().head(10).plot(kind='barh', ax=ax2, color='#10b981')
    ax2.set_title('Top 10 Reported Defect Components')
    ax2.invert_yaxis()

plt.tight_layout()
fig.savefig(output_dir / 'top_makes_and_components.png', dpi=150)
plt.show()


In [ ]:
# 5. Physical Hazard Indicators Analysis
hazard_cols = [c for c in ['crash', 'fire', 'injured', 'deaths'] if c in df.columns]
if hazard_cols:
    hazard_counts = df[hazard_cols].sum()
    print('Hazard Indicator Counts:\n', hazard_counts)
    
    fig, ax = plt.subplots(figsize=(7, 4))
    hazard_counts.plot(kind='bar', ax=ax, color=['#ef4444', '#f97316', '#eab308', '#dc2626'])
    ax.set_title('Severe Safety Hazard Incidents in Fleet')
    ax.set_ylabel('Number of Reported Incidents')
    plt.tight_layout()
    fig.savefig(output_dir / 'safety_hazards.png', dpi=150)
    plt.show()


In [ ]:
# 6. Generate and Save Summary Report
summary = {
    'total_records': len(df),
    'mean_char_length': round(float(df['char_length'].mean()), 2),
    'median_word_count': round(float(df['word_count'].median()), 2),
    'top_components': df['component'].value_counts().head(5).to_dict() if 'component' in df.columns else {},
    'hazard_counts': {c: int(df[c].sum()) for c in hazard_cols},
}

(output_dir / 'eda_summary.json').write_text(json.dumps(summary, indent=2))
print('Summary saved:', json.dumps(summary, indent=2))


In [ ]:
# 7. Automated Output Packaging & Browser Download
archive_name = 'autovoice_eda_artifacts'
shutil.make_archive(archive_name, 'zip', output_dir)
print(f'📦 Packaged EDA outputs into {archive_name}.zip')

if IN_COLAB:
    print('⬇️ Downloading artifact archive to your computer...')
    files.download(f'{archive_name}.zip')
else:
    print(f'Artifact archive available at: {Path(archive_name + ".zip").resolve()}')
